# Qwen3-8B ecological numerical-threshold evaluation

This is an **evaluation-only** Colab workflow. It selects one of five existing, hash-verified Qwen3-8B LoRA checkpoints: ecological dilemma prompt-only, ecological-option response-only, human-option response-only, CLASH prompt-only, or CLASH exact-action response-only. It never starts fine-tuning.

Each of the eight `extreme_v2_numeric` scenarios is presented once. The model must choose the largest tolerable human-death count from `0, 1, 10, 100, 1000, 10000, 100000, 1000000`. The workflow scores every exact numerical continuation and normalizes jointly across the complete candidate set, producing a threshold distribution for the unchanged base and selected adapter. It does not run the earlier primary or control question suites.

A new result is completed under local `/content`, copied beneath the selected source run in Google Drive, flushed, remounted, and rehashed before publication. Use an A100 40 GB (or larger). If GitHub publication remains enabled, add a Colab secret named `GITHUB_TOKEN` with Contents read/write permission for `shengweiming/value-misalignment`.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
REPOSITORY_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
stale_project_modules = [
    name for name in tuple(sys.modules)
    if name == "scripts" or name.startswith("scripts.")
]
for module_name in stale_project_modules:
    del sys.modules[module_name]
importlib.invalidate_caches()
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
assert torch.cuda.is_bf16_supported(), "This evaluation requires a BF16-capable GPU."
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 38, "Select an A100 40 GB (or larger) runtime."
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")

Choose one checkpoint below. The configuration reproduces its training signature only so the notebook can find and reverify the existing Drive run. Changing `CHECKPOINT`, numerical candidates, batch size, or publication settings never starts training. `FORCE_EVALUATION=False` reuses a prior numerical bundle only when the source completion hash, exact rendered prompts, candidate grid, protocol, and every result hash match.

In [ ]:
from google.colab import userdata
from scripts.ecological_prompt_sft import DilemmaSFTConfig

CHECKPOINT = "ecological_prompt_only"
# ecological_prompt_only | ecological_option | human_option | clash_prompt_only | clash_action

CHECKPOINT_SPECS = {
    "ecological_prompt_only": {
        "training_arm": "prompt_only",
        "dataset_path": Path("data/ecological_dilemmas/v1/records.jsonl"),
        "pair_name": None,
        "output_slug": "ecological_dilemma_prompt_qwen3_8b",
    },
    "ecological_option": {
        "training_arm": "ecological_option",
        "dataset_path": Path("data/ecological_dilemmas/sft/ecological_option/records.jsonl"),
        "pair_name": None,
        "output_slug": "ecological_dilemma_ecological_option_qwen3_8b",
    },
    "human_option": {
        "training_arm": "human_option",
        "dataset_path": Path("data/ecological_dilemmas/sft/human_option/records.jsonl"),
        "pair_name": None,
        "output_slug": "ecological_dilemma_human_option_qwen3_8b",
    },
    "clash_prompt_only": {
        "training_arm": "prompt_only",
        "dataset_path": Path("data/control_dilemmas/clash/v1/records.jsonl"),
        "pair_name": "qwen3_8b_clash_prompt_control_sft",
        "output_slug": "clash_prompt_control_qwen3_8b",
    },
    "clash_action": {
        "training_arm": "action",
        "dataset_path": Path("data/control_dilemmas/clash/sft/action/records.jsonl"),
        "pair_name": "qwen3_8b_clash_action_sft",
        "output_slug": "clash_action_qwen3_8b",
    },
}
assert CHECKPOINT in CHECKPOINT_SPECS
SPEC = CHECKPOINT_SPECS[CHECKPOINT]
NUMERIC_VALUES = (0, 1, 10, 100, 1_000, 10_000, 100_000, 1_000_000)
DRIVE_OUTPUT_ROOT = (
    Path("/content/drive/MyDrive/value-misalignment") / SPEC["output_slug"]
)
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"

CONFIG = DilemmaSFTConfig(
    output_root=Path("/content/evaluation-only-no-training"),
    training_arm=SPEC["training_arm"],
    dataset_path=SPEC["dataset_path"],
    pair_name=SPEC["pair_name"],
    base_model="Qwen/Qwen3-8B",
    model_revision="b968826d9c46dd6066d109eabc6255188de91218",
    max_length=1024,
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    eval_batch_size=4,
    seed=42,
    cost_counts=NUMERIC_VALUES,
)

GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Add a Colab secret named GITHUB_TOKEN and grant this notebook access before evaluation."
        ) from exc
    if not GITHUB_TOKEN:
        raise RuntimeError("GITHUB_TOKEN is empty; publication is enabled.")

print("Selected checkpoint:", CHECKPOINT)
print("Drive root:", DRIVE_OUTPUT_ROOT)
print("Numerical candidates:", NUMERIC_VALUES)
CONFIG

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
from transformers import AutoTokenizer
from scripts.ecological_prompt_sft import (
    EXTREME_V2_NUMERIC_TEMPLATES,
    build_numeric_threshold_cases,
)
from scripts.harmony_eval.scoring import format_causal_prompt

numeric_preview_cases = build_numeric_threshold_cases(NUMERIC_VALUES)
assert len(numeric_preview_cases) == len(EXTREME_V2_NUMERIC_TEMPLATES) == 8
assert all(len(case["candidates"]) == len(NUMERIC_VALUES) for case in numeric_preview_cases)
print(f"Reviewing all {len(numeric_preview_cases)} numerical prompts before inference.")
for case in numeric_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

preview_tokenizer = AutoTokenizer.from_pretrained(
    CONFIG.base_model,
    revision=CONFIG.model_revision,
    use_fast=True,
)
formatted_preview = format_causal_prompt(
    preview_tokenizer,
    numeric_preview_cases[0]["prompt"],
    enable_thinking=False,
)
prompt_ids = preview_tokenizer.encode(formatted_preview, add_special_tokens=False)
candidate_token_audit = []
for value in NUMERIC_VALUES:
    scored_text = str(value) + preview_tokenizer.eos_token
    full_ids = preview_tokenizer.encode(
        formatted_preview + scored_text,
        add_special_tokens=False,
    )
    assert full_ids[:len(prompt_ids)] == prompt_ids
    candidate_ids = full_ids[len(prompt_ids):]
    assert candidate_ids
    candidate_token_audit.append({
        "candidate_value": value,
        "scored_text": scored_text,
        "candidate_token_count_including_eos": len(candidate_ids),
        "candidate_token_ids": candidate_ids,
    })
display(pd.DataFrame(candidate_token_audit))
print("Exact number-plus-EOS tokenization audit passed.")

In [ ]:
import json
from scripts.ecological_prompt_sft import (
    find_compatible_complete_run,
    validate_complete_run,
)

artifacts = find_compatible_complete_run(DRIVE_OUTPUT_ROOT, CONFIG)
if artifacts is None:
    raise RuntimeError(
        f"No compatible hash-verified {CHECKPOINT} checkpoint was found under {DRIVE_OUTPUT_ROOT}. "
        "This evaluation-only notebook will not retrain it."
    )
validate_complete_run(artifacts)
run_metadata = json.loads(artifacts.metadata_path.read_text())
assert run_metadata["training_objective"]
assert run_metadata["config"]["base_model"] == CONFIG.base_model
print("Verified source run:", artifacts.run_dir)
print("Training objective:", run_metadata["training_objective"])
print("Final adapter:", artifacts.final_adapter_dir)

In [ ]:
from scripts.ecological_prompt_sft import run_numeric_threshold_workflow

numeric_workflow = run_numeric_threshold_workflow(
    artifacts,
    cost_counts=NUMERIC_VALUES,
    batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION,
)
if numeric_workflow.evaluation_reused:
    print("NUMERIC EVALUATION SKIPPED — reusing the matching verified Drive bundle.")
else:
    print("NUMERIC EVALUATION COMPLETE — verified after a fresh Drive remount.")
print("Verified Drive result:", numeric_workflow.evaluation_artifacts.output_dir)
numeric_workflow.validation

In [ ]:
from IPython.display import Image

numeric_artifacts = numeric_workflow.evaluation_artifacts
numeric_scores = pd.read_csv(numeric_artifacts.raw_scores_path)
numeric_summaries = pd.read_csv(numeric_artifacts.thresholds_path)
assert len(numeric_scores) == 2 * len(numeric_preview_cases) * len(NUMERIC_VALUES)
assert set(numeric_scores["model_role"]) == {"base", "aligned"}
probability_table = numeric_scores.pivot(
    index=["template_family", "candidate_value"],
    columns="model_role",
    values="candidate_probability",
).sort_index()
summary_table = numeric_summaries.pivot(
    index="template_family",
    columns="model_role",
    values=[
        "mode_threshold",
        "median_threshold",
        "expected_log1p_threshold",
        "entropy_nats",
        "probability_threshold_zero",
    ],
).sort_index()
display(probability_table)
display(summary_table)
display(Image(filename=str(numeric_artifacts.plot_path)))
print("Rendered cases:", numeric_artifacts.rendered_cases_path)
print("Raw candidate scores:", numeric_artifacts.raw_scores_path)
print("Distribution summaries:", numeric_artifacts.thresholds_path)
print("Completion manifest:", numeric_artifacts.complete_marker_path)

In [ ]:
from scripts.ecological_prompt_sft import publish_results_to_github

publication = None
if PUBLISH_TO_GITHUB:
    publication = publish_results_to_github(
        numeric_workflow.evaluation_artifacts,
        source_run_name=artifacts.run_dir.name,
        github_repository=GITHUB_REPOSITORY,
        branch=GITHUB_BRANCH,
        github_token=GITHUB_TOKEN,
        repo_root=REPO_DIR,
    )
    print("GitHub publication verified:", publication.html_url)
    print("GitHub commit:", publication.commit_sha)
else:
    print("GitHub publication disabled; the verified Drive bundle is intact.")

A successful run verifies three things independently: the selected source checkpoint matches its dataset, objective, pair identity, model and LoRA signature, and original artifact hashes; the numerical bundle contains exactly eight candidates for each of eight scenarios and both model roles, with every within-case distribution summing to one; and the Drive copy passes a fresh-mount hash check. If publication is enabled, the GitHub branch tip is also read back after the non-force push. No step retrains an adapter, runs the legacy control questions, overwrites a prior result, or places the GitHub token in repository state.